# 漂移探索：先用眼睛看，再交給統計檢定

> **這本為什麼是 notebook**：漂移分析的本質**就是資料分析**——人盯著分布看，
> 調參數、換欄位、問「這樣算漂移嗎」。Evidently 本來就產 HTML 報告，
> 原本要存檔再切到瀏覽器開；在 notebook 裡直接看，少一次 context switch。
>
> **配套腳本**：同資料夾的 `drift_report.py` 是**同一套邏輯的腳本版**，給 CI 無人值守跑。
> 這是 m6 刻意並存兩種介質的地方：
>
> ```
> drift_explore.ipynb   ← 人在看：調參數、觀察分布怎麼被推開、決定門檻
> drift_report.py       ← 機器在跑：把結論固定下來，每天自動檢查
> ```
>
> 一句話：**你在 notebook 想清楚要監控什麼，然後寫成腳本讓機器每天替你看。**

In [ ]:
from pathlib import Path


def find_course_root() -> Path:
    """從當前目錄往上找到 mlops-course 根目錄（含 datasets/ 的那層）。

    notebook 沒有 __file__，而且你可能從任何位置啟動 Jupyter，
    所以用「往上層找標記檔」定位，不寫死相對路徑。
    """
    for base in [Path.cwd(), *Path.cwd().parents]:
        if (base / "datasets" / "iris.csv").exists():
            return base
    raise FileNotFoundError("找不到 mlops-course/datasets/，請在 mlops-course/ 之內開啟本 notebook")


ROOT = find_course_root()
print("course root =", ROOT)

import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")  # Evidently 的版本提示訊息會蓋掉輸出，這裡關掉

SEED = 42
SENSOR_COLS = ["temperature", "vibration", "current"]

sensors = pd.read_csv(ROOT / "datasets" / "toy_sensors.csv", parse_dates=["event_timestamp"])
print(f"{len(sensors)} 列、{sensors.machine_id.nunique()} 台機器")
sensors.head()

## 1. 切 reference / current —— 這一步最容易做錯

- **reference**：模型上線當時、訓練資料的分布長相
- **current**：線上現在進來的新資料

這是個**時間**的切法。但 `toy_sensors.csv` 是「一台機器接一台機器」排序的，
**照列序對半切 = 照機台切**，而不同機台的溫度基準本來就不一樣：

In [ ]:
half = len(sensors) // 2
print("列序前半的機台：", sorted(sensors.iloc[:half].machine_id.unique()))
print("列序後半的機台：", sorted(sensors.iloc[half:].machine_id.unique()))
print()
print("各機台平均溫度：")
print(sensors.groupby("machine_id").temperature.mean().round(2).to_string())

機台之間差了好幾度。**照列序切，你比較的是「不同機器」而不是「不同時間」**——
監控會從第一天就在報假警。

正確做法：**先按時間排序再切**，讓所有機台都出現在兩邊。

In [ ]:
ordered = sensors.sort_values("event_timestamp").reset_index(drop=True)
cut = len(ordered) // 2

reference = ordered.iloc[:cut][SENSOR_COLS].reset_index(drop=True)
current_clean = ordered.iloc[cut:][SENSOR_COLS].reset_index(drop=True)

print(f"reference：{len(reference)} 列，{ordered.event_timestamp.iloc[0]} → {ordered.event_timestamp.iloc[cut - 1]}")
print(f"current  ：{len(current_clean)} 列，{ordered.event_timestamp.iloc[cut]} → {ordered.event_timestamp.iloc[-1]}")
print()
print("兩邊的機台組成（應該一致）：")
print("  reference:", sorted(ordered.iloc[:cut].machine_id.unique()))
print("  current  :", sorted(ordered.iloc[cut:].machine_id.unique()))

## 2. 先用眼睛看：還沒注入漂移時的分布

**沒有漂移時兩條分布應該高度重疊**。這是你的「正常長相」基準。

In [ ]:
def plot_overlay(ref: pd.DataFrame, cur: pd.DataFrame, title: str):
    """把 reference / current 的分布疊在一起畫，一欄一張。"""
    fig, axes = plt.subplots(1, len(SENSOR_COLS), figsize=(13, 3.4))
    for ax, col in zip(axes, SENSOR_COLS):
        ax.hist(ref[col], bins=25, alpha=0.6, label="reference", color="#4C78A8", density=True)
        ax.hist(cur[col], bins=25, alpha=0.6, label="current", color="#F58518", density=True)
        ax.set_title(col)
        ax.set_xlabel("value")
    axes[0].set_ylabel("density")
    axes[0].legend(fontsize=8)
    fig.suptitle(title)
    plt.tight_layout()
    plt.show()


plot_overlay(reference, current_clean, "Before injection: distributions should overlap")

## 3. 注入漂移，再看一次

模擬**感測器老化**：溫度整體往上平移。

In [ ]:
def inject_drift(frame: pd.DataFrame, column: str, shift: float) -> pd.DataFrame:
    """回傳新的 DataFrame（不可變寫法），把指定欄位整體平移 shift。"""
    drifted = frame.copy()
    drifted[column] = drifted[column] + shift
    return drifted


shift_amount = 3.0 * reference["temperature"].std()
current_drifted = inject_drift(current_clean, "temperature", shift_amount)

print(f"對 temperature 平移 +{shift_amount:.2f}（= 3 倍標準差，模擬感測器老化）")
plot_overlay(reference, current_drifted, "After injection: temperature is pushed apart")

肉眼就看得出 temperature 整條被推開了，另外兩欄沒動。
接下來看統計檢定會不會同意你的眼睛。

## 4. 交給 Evidently 做統計判定

In [ ]:
from evidently.metric_preset import DataDriftPreset
from evidently.report import Report


def drift_summary(ref: pd.DataFrame, cur: pd.DataFrame) -> tuple[dict, pd.DataFrame]:
    """跑一次 DataDriftPreset，回傳（整體結論, 逐欄明細）。"""
    report = Report(metrics=[DataDriftPreset()])
    report.run(reference_data=ref, current_data=cur)
    payload = report.as_dict()

    overall, per_column = {}, []
    for metric in payload["metrics"]:
        if metric["metric"] == "DatasetDriftMetric":
            overall = metric["result"]
        if metric["metric"] == "DataDriftTable":
            for col, info in metric["result"]["drift_by_columns"].items():
                per_column.append({
                    "column": col,
                    "檢定方法": info["stattest_name"],
                    "drift_score": round(float(info["drift_score"]), 6),
                    "偵測到漂移": info["drift_detected"],
                })
    return overall, pd.DataFrame(per_column).set_index("column")


overall, per_col = drift_summary(reference, current_drifted)
print(f"逐欄：{overall['number_of_drifted_columns']} / {overall['number_of_columns']} 欄被標記為漂移")
print(f"整體：dataset_drift = {overall['dataset_drift']}"
      f"（門檻 drift_share = {overall['drift_share']}，"
      f"實際漂移比例 = {overall['share_of_drifted_columns']:.2f}）")
per_col

統計檢定同意你的眼睛：**只有 temperature 被標記為漂移**，另外兩欄沒有。

### 但請注意整體旗標是 `False`

`dataset_drift` 預設要**超過半數欄位**都漂移才亮紅燈（`drift_share = 0.5`）。
這裡 1/3 ≈ 0.33 沒過門檻，所以整體結論是 `False`——**即使 temperature 已經明顯漂掉**。

這是實務上很容易踩的坑：

> **只看整體旗標會漏掉單欄漂移。**
> 要嘛看逐欄明細，要嘛依你的風險承受度把 `drift_share` 調低。

一個關鍵欄位漂掉往往就足以讓模型失效，不需要等到半數欄位一起漂。
這也是為什麼 `drift_report.py` 在腳本版裡要明確決定「用哪個層級當 CI 的門檻」。

**對照組**——沒注入漂移時，應該是乾淨的（這就是為什麼 §1 的切法很重要）：

In [ ]:
clean_overall, clean_per_col = drift_summary(reference, current_clean)
print(f"未注入漂移時 dataset_drift = {clean_overall['dataset_drift']}"
      f"（漂移欄位數 = {clean_overall['number_of_drifted_columns']}）")
clean_per_col

## 5. 存一份完整 HTML 報告

Evidently 的 HTML 報告包含每欄的分布圖與檢定細節，適合當**事件記錄**存檔或寄給同事。

In [ ]:
report = Report(metrics=[DataDriftPreset()])
report.run(reference_data=reference, current_data=current_drifted)

out_path = Path.cwd() / "drift_explore_report.html"  # 命名對齊 .gitignore 的 *_report.html
report.save_html(str(out_path))
print(f"已輸出：{out_path}")
print("（在 Jupyter 裡也可以直接跑 report.show() 內嵌顯示）")

## 6. 實驗：多大的漂移才抓得到？

**這是 notebook 最有價值的地方**——反覆調一個參數看結果怎麼變。
把平移量從 0 拉到 3 倍標準差，找出偵測門檻在哪。

In [ ]:
rows = []
for k in [0.0, 0.25, 0.5, 0.75, 1.0, 1.5, 2.0, 3.0]:
    shifted = inject_drift(current_clean, "temperature", k * reference["temperature"].std())
    _, detail = drift_summary(reference, shifted)
    rows.append({
        "平移量（倍標準差）": k,
        "drift_score": detail.loc["temperature", "drift_score"],
        "偵測到漂移": detail.loc["temperature", "偵測到漂移"],
    })

sweep = pd.DataFrame(rows)
sweep

In [ ]:
# p-value 會小到浮點數 underflow 成 0，直接取 log 會變 -inf。
# 取一個下限截斷，並改畫 -log10(p)：柱子越高＝越確定有漂移，比 log 軸好讀。
FLOOR = 1e-16
sweep["-log10(p)"] = -np.log10(np.clip(sweep["drift_score"], FLOOR, 1.0))

fig, ax = plt.subplots(figsize=(8, 3.6))
colors = ["#E45756" if d else "#54A24B" for d in sweep["偵測到漂移"]]
ax.bar(sweep["平移量（倍標準差）"].astype(str), sweep["-log10(p)"], color=colors, width=0.6)
ax.axhline(-np.log10(0.05), ls="--", color="grey", lw=1)
ax.text(0.02, -np.log10(0.05) + 0.6, "p = 0.05 threshold",
        fontsize=8, color="grey", transform=ax.get_yaxis_transform())
ax.set_xlabel("injected shift (x std)")
ax.set_ylabel("-log10(K-S p-value)")
ax.set_title("How much drift is detectable?  (red = flagged, capped at 1e-16)")
plt.tight_layout()
plt.show()

讀圖：柱子越過灰線就被標記為漂移。

**注意靈敏度**：只要平移 **0.25 倍標準差**就已經被抓到了。
K-S 檢定在樣本數大時非常敏感——這意味著在生產環境，**它很可能每天都在對你尖叫**。

這正是需要人做判斷的地方：

- 統計上顯著 ≠ 業務上重要。0.25 個標準差的溫度變化，模型可能根本不在乎。
- 實務常見對策：改用對「幅度」敏感的指標（PSI、Wasserstein），或加上「連續 N 天才告警」。

**這個門檻決策需要人看著圖決定，決定完才寫進腳本。**

## 7. 從 notebook 到 CI

你剛剛完成的是**人的工作**：決定監控哪幾欄、確認 reference 切得對、找出偵測門檻。

接下來把結論固定成**機器的工作**：

```bash
python drift_report.py      # 同一套邏輯，給 CI 每天自動跑
```

| | `drift_explore.ipynb`（本檔） | `drift_report.py` |
| :--- | :--- | :--- |
| 誰在用 | 人，互動式 | 機器，無人值守 |
| 目的 | 決定「監控什麼、門檻多少」 | 執行既定的檢查、輸出結論 |
| 產出 | 理解 | 報告檔 + 非零 exit code |

> 這就是課程反覆出現的節奏，而 m6 是唯一一個**兩種介質並存**的模組——
> 因為監控天生同時需要「人的判斷」與「機器的重複執行」。

**閉環**：漂移偵測到之後呢？回到 m5，把它接成
「漂移告警 → 觸發 CT 重訓」。監控不是終點，是自動化的觸發器。